# Análisis exploratorio de Oxford-IIIT Pet

In [ ]:
# Importar bibiotecas necesarias
from pathlib import Path				# Para manejar rutas de archivos y directorios
import numpy as np						# Para procesar mascaras con etiquetas numéricas
from PIL import Image					# Para procesar imágenes
import json								# Para procesar archivos JSON
import xml.etree.ElementTree as ET 		# Para procesar archivos XML
import pandas as pd						# Para procesar archivos CSV
from matplotlib import pyplot as plt	# Para visualizar imágenes y máscaras
from matplotlib.patches import Patch	# Para crear leyendas personalizadas en los gráficos

In [ ]:
# Leer el archivo de configuración
PROYECTO = Path("..")
ruta_config = PROYECTO / "config" / "dataset.json"
with ruta_config.open("r", encoding="utf-8") as archivo:
	config = json.load(archivo)

# Definir rutas de archivos y directorios
RUTA_MANIFEST = PROYECTO / config["manifest"]["path"]
DATA = PROYECTO / config["dataset"]["root"]
IMAGES = DATA / config["dataset"]["images"]
MASKS = DATA / config["dataset"]["masks"]

# Cargar el archivo manifest
manifest = pd.read_csv(RUTA_MANIFEST)
manifest.head()

## Análisis de categorías

In [ ]:
conteos = (
    manifest.groupby(["species", "breed"])
    .size()
    .reset_index(name="n_images")
    .sort_values(["species", "breed"])
)

colores = {
    "cat": "#568A80",
    "dog": "#D0805C"
}

fig, axes = plt.subplots(
    1, 2,
    figsize=(15, 10),
    sharex=True
)

for ax, especie, titulo in zip(
    axes,
    ["cat", "dog"],
    ["Gatos", "Perros"]
):
    datos = conteos[
        conteos["species"] == especie
    ].sort_values("n_images")

    ax.barh(
        datos["breed"],
        datos["n_images"],
        color=colores[especie]
    )

    ax.set_title(titulo, fontsize=13, fontweight="bold")
    ax.set_xlabel("Número de imágenes")
    ax.set_xlim(0, 220)
    ax.grid(axis="x", alpha=0.2)
    ax.set_axisbelow(True)

    for i, valor in enumerate(datos["n_images"]):
        ax.text(
            valor + 2,
            i,
            str(valor),
            va="center",
            fontsize=8
        )

fig.suptitle(
    "Distribución de imágenes por raza",
    fontsize=15,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

In [ ]:
# Contar imágenes por especie y raza
conteos = (
    manifest.groupby(["species", "breed"])
    .size()
    .sort_index()
)

# Totales por especie
especies = conteos.groupby(level="species").sum()

# Colores principales
colores_especies = {
    "cat": "#568A80",
    "dog": "#D0805C"
}

# Crear variaciones de color para las razas
paletas = {
    "cat": plt.cm.Greens,
    "dog": plt.cm.Oranges
}

colores_razas = {}

for especie in especies.index:

    razas = conteos.loc[especie].index
    valores = np.linspace(0.35, 0.85, len(razas))

    for raza, valor in zip(razas, valores):
        colores_razas[(especie, raza)] = paletas[especie](valor)

# Preparar colores de ambos anillos
colores_exteriores = [
    colores_razas[(especie, raza)]
    for especie, raza in conteos.index
]

colores_interiores = [
    colores_especies[especie]
    for especie in especies.index
]

# Crear figura
fig, ax = plt.subplots(figsize=(12, 9))

# Anillo exterior: razas
ax.pie(
    conteos.values,
    radius=1.0,
    colors=colores_exteriores,
    startangle=90,
    counterclock=False,
    wedgeprops={
        "width": 0.3,
        "edgecolor": "white",
        "linewidth": 0.6
    }
)

# Anillo interior: especies
ax.pie(
    especies.values,
    radius=0.68,
    colors=colores_interiores,
    startangle=90,
    counterclock=False,
    autopct=lambda p: f"{p:.2f}%",
    pctdistance=0.76,
    textprops={
        "color": "white",
        "weight": "bold",
        "fontsize": 11
    },
    wedgeprops={
        "width": 0.3,
        "edgecolor": "white",
        "linewidth": 2
    }
)

# Total en el centro
ax.text(
    0, 0,
    f"{len(manifest):,}\nimágenes",
    ha="center",
    va="center",
    fontsize=13,
    fontweight="bold",
    color="#2C3431"
)

# Leyenda por especie y raza
leyenda = [
    Patch(
        facecolor=colores_razas[(especie, raza)],
        label=f"{raza.replace('_', ' ')} ({cantidad})"
    )
    for (especie, raza), cantidad in conteos.items()
]

ax.legend(
    handles=leyenda,
    title="Razas",
    loc="center left",
    bbox_to_anchor=(1.05, 0.5),
    fontsize=8,
    title_fontsize=10,
    ncol=2,
    frameon=False
)

ax.set_title(
    "Distribución de imágenes por especie y raza",
    fontsize=15,
    fontweight="bold",
    pad=20
)

ax.set_aspect("equal")

plt.tight_layout()
plt.show()

## Análisis de tamaño

### Dispersión de ancho frente a alto

In [ ]:
import matplotlib.pyplot as plt

colores = {
    "cat": "#568A80",
    "dog": "#D0805C"
}

etiquetas = {
    "cat": "Gatos",
    "dog": "Perros"
}

fig, ax = plt.subplots(figsize=(8, 7))

for especie, grupo in manifest.groupby("species"):

    ax.scatter(
        grupo["width"],
        grupo["height"],
        label=etiquetas[especie],
        color=colores[especie],
        alpha=0.25,
        s=20,
        edgecolors="none"
    )

# Línea de referencia: ancho = alto
limite = max(
    manifest["width"].max(),
    manifest["height"].max()
)

ax.plot(
    [0, limite],
    [0, limite],
    linestyle="--",
    color="#2C3431",
    linewidth=1.5,
    label="Ancho = Alto"
)

ax.set_xlim(0, limite * 1.05)
ax.set_ylim(0, limite * 1.05)

ax.set_aspect("equal", adjustable="box")

ax.set_xlabel("Ancho (píxeles)")
ax.set_ylabel("Alto (píxeles)")
ax.set_title("Distribución de dimensiones por especie")

ax.legend()
ax.grid(True, alpha=0.2)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

### Relación de aspecto (ancho/alto)

In [ ]:
# Calcular relación de aspecto
manifest["aspect_ratio"] = manifest["width"] / manifest["height"]

# Crear histograma
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(
    manifest["aspect_ratio"],
    bins=15,
    color="#568A80",
    edgecolor="white",
    alpha=0.85
)

# Línea de referencia para imágenes cuadradas
ax.axvline(
    x=1,
    color="#D0805C",
    linestyle="--",
    linewidth=2,
    label="Relación 1:1 (cuadrada)"
)

ax.set_xlabel("Relación de aspecto (ancho / alto)")
ax.set_ylabel("Número de imágenes")
ax.set_title("Distribución de relaciones de aspecto")

ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()
print(manifest["aspect_ratio"].describe().round(2))

## Análisis de máscaras

In [ ]:
def analizar_mascara(ruta_mascara):

    with Image.open(ruta_mascara) as img:
        mascara = np.array(img)

    # Contar píxeles por clase
    pixeles_animal = np.count_nonzero(mascara == 1)
    pixeles_fondo = np.count_nonzero(mascara == 2)
    pixeles_contorno = np.count_nonzero(mascara == 3)

    pixeles_totales = mascara.size

    return {
        "pixels_total": pixeles_totales,
        "pixels_animal": pixeles_animal,
        "pixels_fondo": pixeles_fondo,
        "pixels_contorno": pixeles_contorno,
        "pct_animal": pixeles_animal / pixeles_totales * 100,
        "pct_fondo": pixeles_fondo / pixeles_totales * 100,
        "pct_contorno": pixeles_contorno / pixeles_totales * 100
    }

In [ ]:
# Analizar todas las máscaras y crear un DataFrame con los resultados
registros = []

for _, fila in manifest.iterrows():

    ruta_mascara = DATA / fila["mask_path"]

    datos = analizar_mascara(ruta_mascara)

    registros.append({
        "image_id": fila["image_id"],
        "breed": fila["breed"],
        "species": fila["species"],
        **datos
    })

df_mascaras = pd.DataFrame(registros)

display(df_mascaras.head())

In [ ]:
# Verificar consistencia de píxeles
df_mascaras["pixels_verificados"] = (
    df_mascaras["pixels_animal"]
    + df_mascaras["pixels_fondo"]
    + df_mascaras["pixels_contorno"]
)

print(
    "Máscaras con etiquetas inesperadas:",
    (df_mascaras["pixels_verificados"] != df_mascaras["pixels_total"]).sum()
)

In [ ]:
# Crear histograma del porcentaje de píxeles del animal
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(
    df_mascaras["pct_animal"],
    bins=12,
    color="#568A80",
    edgecolor="white",
    alpha=0.85
)

ax.axvline(
    df_mascaras["pct_animal"].median(),
    color="#D0805C",
    linestyle="--",
    linewidth=2,
    label=f"Mediana: {df_mascaras['pct_animal'].median():.1f}%"
)

ax.set_xlabel("Porcentaje de píxeles del animal (%)")
ax.set_ylabel("Número de imágenes")
ax.set_title("Distribución del área ocupada por el animal")

ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Crear gráfico de barras apiladas del porcentaje de píxeles por clase
df_ordenado = df_mascaras.sort_values("pct_animal")

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(df_ordenado))

ax.bar(
    x,
    df_ordenado["pct_animal"],
    label="Animal",
    color="#568A80"
)

ax.bar(
    x,
    df_ordenado["pct_fondo"],
    bottom=df_ordenado["pct_animal"],
    label="Fondo",
    color="#93A3C5"
)

ax.bar(
    x,
    df_ordenado["pct_contorno"],
    bottom=(
        df_ordenado["pct_animal"]
        + df_ordenado["pct_fondo"]
    ),
    label="Contorno",
    color="#D0805C"
)

ax.set_xlabel("Imágenes ordenadas por área del animal")
ax.set_ylabel("Porcentaje de píxeles (%)")
ax.set_title("Distribución de clases en cada máscara")

ax.set_ylim(0, 100)
ax.set_xticks([])
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Crear gráfico de pastel con la distribución global de píxeles por clase
totales = df_mascaras[
    ["pixels_animal", "pixels_fondo", "pixels_contorno"]
].sum()

fig, ax = plt.subplots(figsize=(7, 6))

ax.pie(
    totales.values,
    labels=["Animal", "Fondo", "Contorno"],
    colors=["#568A80", "#93A3C5", "#D0805C"],
    autopct="%1.1f%%",
    startangle=90,
    wedgeprops={
        "edgecolor": "white",
        "linewidth": 2
    }
)

ax.set_title("Distribución global de píxeles por clase")

plt.tight_layout()
plt.show()

In [ ]:
# Crear gráfico de violín para el número de píxeles del animal

datos = df_mascaras["pixels_animal"].dropna().to_numpy()

# Estadísticas descriptivas
media = np.mean(datos)
mediana = np.median(datos)
q1, q3 = np.percentile(datos, [25, 75])

# Crear figura
fig, ax = plt.subplots(figsize=(7, 7))

# Violin plot
partes = ax.violinplot(
    datos,
    positions=[1],
    widths=0.65,
    showmeans=False,
    showmedians=False,
    showextrema=False
)

# Estilo del violín
for cuerpo in partes["bodies"]:
    cuerpo.set_facecolor("#568A80")
    cuerpo.set_edgecolor("#2C3431")
    cuerpo.set_alpha(0.40)
    cuerpo.set_zorder(1)

# Observaciones individuales:
# desplazadas hacia la derecha del violín
rng = np.random.default_rng(42)

x = 1.48 + rng.uniform(
    -0.10, 0.10,
    size=len(datos)
)

ax.scatter(
    x, datos,
    color="#568A80",
    s=5,
    alpha=0.08,
    edgecolors="none",
    zorder=2,
    label="Imágenes individuales"
)

# Rango intercuartílico Q1-Q3
ax.vlines(
    1, q1, q3,
    color="#2C3431",
    linewidth=9,
    zorder=5,
    label="Rango intercuartílico (Q1-Q3)"
)

# Cuartiles Q1 y Q3
ax.scatter(
    [1, 1],
    [q1, q3],
    color="#2C3431",
    s=55,
    zorder=6
)

# Mediana
ax.hlines(
    mediana,
    xmin=0.83,
    xmax=1.17,
    color="#CD972B",
    linewidth=3,
    zorder=7,
    label=f"Mediana: {mediana:,.0f} px"
)

# Media
ax.scatter(
    1, media,
    color="#D0805C",
    marker="D",
    s=90,
    edgecolor="white",
    linewidth=1,
    zorder=8,
    label=f"Media: {media:,.0f} px"
)

# Configuración
ax.set_xticks([1])
ax.set_xticklabels(["Animal"])

ax.set_ylabel("Número de píxeles del animal")
ax.set_title("Distribución del tamaño absoluto del animal")

ax.set_xlim(0.55, 1.7)
ax.set_ylim(bottom=0)

ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)

ax.legend(
    loc="upper left",
    bbox_to_anchor=(1.02, 1)
)

plt.tight_layout()
plt.show()

# Estadísticas descriptivas
print(f"Número de máscaras: {len(datos)}")
print(f"Media:   {media:,.0f} píxeles")
print(f"Mediana: {mediana:,.0f} píxeles")
print(f"Q1:      {q1:,.0f} píxeles")
print(f"Q3:      {q3:,.0f} píxeles")
print(f"RIC:     {q3 - q1:,.0f} píxeles")

In [ ]:
# Crear gráfico de violín para el porcentaje de píxeles del animal

datos = df_mascaras["pct_animal"].dropna().to_numpy()

# Estadísticas descriptivas
media = np.mean(datos)
mediana = np.median(datos)
q1, q3 = np.percentile(datos, [25, 75])

fig, ax = plt.subplots(figsize=(7, 7))

# Violin plot
partes = ax.violinplot(
    datos,
    positions=[1],
    widths=0.65,
    showmeans=False,
    showmedians=False,
    showextrema=False
)

# Ajustar color del violín
for cuerpo in partes["bodies"]:
    cuerpo.set_facecolor("#568A80")
    cuerpo.set_edgecolor("#2C3431")
    cuerpo.set_alpha(0.40)
    cuerpo.set_zorder(1)

# Observaciones individuales:
# desplazadas hacia la derecha del violín
rng = np.random.default_rng(42)

x = 1.48 + rng.uniform(
    -0.10, 0.10,
    size=len(datos)
)

ax.scatter(
    x, datos,
    color="#568A80",
    s=5,
    alpha=0.08,
    edgecolors="none",
    zorder=2,
    label="Imágenes individuales"
)

# Rango intercuartílico Q1-Q3
ax.vlines(
    1, q1, q3,
    color="#2C3431",
    linewidth=9,
    zorder=5,
    label="Rango intercuartílico (Q1-Q3)"
)

# Cuartiles Q1 y Q3
ax.scatter(
    [1, 1],
    [q1, q3],
    color="#2C3431",
    s=55,
    zorder=6
)

# Mediana
ax.hlines(
    mediana,
    xmin=0.83,
    xmax=1.17,
    color="#CD972B",
    linewidth=3,
    zorder=7,
    label=f"Mediana: {mediana:.2f}%"
)

# Media
ax.scatter(
    1, media,
    color="#D0805C",
    marker="D",
    s=90,
    edgecolor="white",
    linewidth=1,
    zorder=8,
    label=f"Media: {media:.2f}%"
)

# Configuración
ax.set_xticks([1])
ax.set_xticklabels(["Animal"])

ax.set_ylabel("Área ocupada por el animal (%)")
ax.set_title("Distribución del tamaño relativo del animal")

ax.set_xlim(0.55, 1.7)
ax.set_ylim(0, 100)

ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)

ax.legend(
    loc="upper left",
    bbox_to_anchor=(1.02, 1)
)

plt.tight_layout()
plt.show()

# Estadísticas
print(f"Número de máscaras: {len(datos)}")
print(f"Media:   {media:.2f}%")
print(f"Mediana: {mediana:.2f}%")
print(f"Q1:      {q1:.2f}%")
print(f"Q3:      {q3:.2f}%")
print(f"RIC:     {q3 - q1:.2f} puntos porcentuales")